<a href="https://colab.research.google.com/github/DonDurkheim/architecture-designs/blob/main/kinyTTS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install dependencies (uncomment and run if needed)
!pip install torch torchaudio huggingface_hub
!pip install git+https://github.com/c4ir-rw/ac-ai-models.git#subdirectory=DeepKIN-AgAI

  Cloning https://github.com/c4ir-rw/ac-ai-models.git to /tmp/pip-req-build-odb8alc0
  Running command git clone --filter=blob:none --quiet https://github.com/c4ir-rw/ac-ai-models.git /tmp/pip-req-build-odb8alc0
  Resolved https://github.com/c4ir-rw/ac-ai-models.git to commit c28db6bfb6b80666583717c87e8edaa2a8f384d5
  Preparing metadata (setup.py) ... done
  Created wheel for deepkin: filename=deepkin-1.0.0-py3-none-any.whl size=184486 sha256=4a0a879e1118c45e777cd43585dd9dfad571455172a2e005880d1636130109cf
  Stored in directory: /tmp/pip-ephem-wheel-cache-tw8_uhyq/wheels/aa/eb/b5/a0546a087db935bdab8ed4466935cd56cb5f1fe9f2bd16dbae
Successfully built deepkin


In [2]:
import torch
import torchaudio
from huggingface_hub import hf_hub_download
from IPython.display import Audio, display

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.10.0+cu128
CUDA available: True


In [3]:
# Download the pretrained model from HuggingFace
model_path = hf_hub_download(
    repo_id="C4IR-RW/kinya-flex-tts",
    filename="kinya_flex_tts_base_trained.pt"
)
print(f"Model downloaded to: {model_path}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


kinya_flex_tts_base_trained.pt:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Model downloaded to: /root/.cache/huggingface/hub/models--C4IR-RW--kinya-flex-tts/snapshots/52147b0cb066535e6357204caa413723a4ba2036/kinya_flex_tts_base_trained.pt


In [4]:
!pip install typed-argument-parser
from deepkin.data.kinya_norm import text_to_sequence
from deepkin.models.flex_tts import FlexKinyaTTS
from deepkin.modules.tts_commons import intersperse

# Define inference device
device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
print(f"Using device: {device}")

# Load TTS model
kinya_tts = FlexKinyaTTS.from_pretrained(device, model_path)
kinya_tts.eval()
print("Model loaded successfully!")

Using device: cuda:0
2026-03-12 20:14:06 Loading FlexTTS from /root/.cache/huggingface/hub/models--C4IR-RW--kinya-flex-tts/snapshots/52147b0cb066535e6357204caa413723a4ba2036/kinya_flex_tts_base_trained.pt
2026-03-12 20:14:12 FlexTTS train steps: 2,000K
2026-03-12 20:14:12 Loading FlexTTS from /root/.cache/huggingface/hub/models--C4IR-RW--kinya-flex-tts/snapshots/52147b0cb066535e6357204caa413723a4ba2036/kinya_flex_tts_base_trained.pt done!
Model loaded successfully!


In [6]:
def synthesize(text: str, speaker_id: int = 0, sampling_rate: int = 24000):
    """Synthesize speech from Kinyarwanda text.

    Args:
        text: Kinyarwanda text to synthesize.
        speaker_id: 0 = Female 1, 1 = Female 2, 2 = Male.
        sampling_rate: Output sampling rate (default 24000).

    Returns:
        Audio widget for playback in the notebook.
    """
    text_id_sequence = intersperse(text_to_sequence(text, norm=True), 0)
    audio_data = kinya_tts(text_id_sequence, speaker_id)
    return Audio(audio_data.squeeze().cpu().numpy(), rate=sampling_rate)

In [9]:
text = "Ushobora kugana ibigo by'ubushakashatsi mu buhinzi cyangwa ubuyobozi bw'inzego zishinzwe ubuhinzi mu karere kawe kugira ngo ubone inama n'ubuyobozi. Na none wareba amahugurwa cyangwa kwihuza n'amakoperative y'abahinzi kugira ngo ubone ubunararibonye n'ubufasha."

# Try all three speakers
for sid, name in [(0, "Female 1"), (1, "Female 2"), (2, "Male")]:
    print(f"\n--- Speaker: {name} (id={sid}) ---")
    display(synthesize(text, speaker_id=sid))


--- Speaker: Female 1 (id=0) ---



--- Speaker: Female 2 (id=1) ---



--- Speaker: Male (id=2) ---


In [1]:
# Try your own text here
custom_text = "Abantu benshi barabizi ariko barabyirengagiza."
display(synthesize(custom_text, speaker_id=2))

NameError: name 'synthesize' is not defined

In [18]:
# Save audio to file
text_to_save = "Muraho neza! Murakaza neza mu Rwanda."
text_id_sequence = intersperse(text_to_sequence(text_to_save, norm=True), 0)
audio_data = kinya_tts(text_id_sequence, 0)

output_path = "output.wav"
torchaudio.save(output_path, audio_data.cpu(), 24000)
print(f"Saved to {output_path}")
display(Audio("output.wav"))

Saved to output.wav
